# HPCG overnight energy estimates — 15 October 2026

CCE 144×1 was the fastest individual diagnostic; 72×1 had the highest wrapper-step efficiency estimate. Neither is a repeated, confirmed optimum. All nine runs are below the official duration requirement.

## Context and method

Join `docs/hpcg/experiment_log.md` (nine VALID HPCG diagnostics) to `analysis/hpcg/overnight-20261015-sacct-energy.psv` by job ID. Use only `.0` joules / `.0` elapsed seconds for average power; divide HPCG GFLOP/s by that power. Keep whole-job energy separate. Never sum overlapping `.batch`, `.extern`, `.0` and allocation records.

Assumptions: exclusive one-node allocations are established by the experiment log. The PSV alone does not include node lists. Step time includes initialisation/setup/validation, so these estimates do not measure the HPCG timed window. One sample per reduced-rank layout and slightly different equation counts limit comparisons.

## Data and checks

Open this notebook from within the repository. It uses only Python's standard library. Source hashes are printed with the results. These cells were executed sequentially with Python 3.14 during local preparation; a Jupyter frontend/kernel was unavailable. The same cells can be rerun in Jupyter with Python 3.11+.


In [1]:
from pathlib import Path
import hashlib, importlib.util
cwd = Path.cwd()
root = next((p for p in (cwd, *cwd.parents) if (p / "analysis/hpcg/overnight_energy.py").exists()), None)
assert root is not None, "Open from within the ciuk-2026-hpcg repository"
spec = importlib.util.spec_from_file_location("overnight_energy", root / "analysis/hpcg/overnight_energy.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
log = root / "docs/hpcg/experiment_log.md"
raw = root / "analysis/hpcg/overnight-20261015-sacct-energy.psv"
rows = analysis.analyse(log.read_text(), raw.read_text())
assert len(rows) == 9
assert sum(r["whole_job_energy_j"] for r in rows) == 2337351
for path in (log, raw):
    print(path.relative_to(root), hashlib.sha256(path.read_bytes()).hexdigest())
print("9/9 matched; no missing/duplicate job IDs; all allocation and .0 records completed.")


docs/hpcg/experiment_log.md 4a1944b79595d5edbac048d851c0dd0a7f04cc1353dc02787349d1030e84180d
analysis/hpcg/overnight-20261015-sacct-energy.psv c4cc65eb50e0dcf0ff19049c4c182df856be71822e4a68bebd1be915bbe89466
9/9 matched; no missing/duplicate job IDs; all allocation and .0 records completed.


In [2]:
print("Job     GFLOP/s  Whole-job J  Step J  Step s  Avg step W  Estimated GFLOP/s/W")
for r in rows:
    print(f"{r['job_id']}  {r['gflops']:7.3f}  {r['whole_job_energy_j']:11.0f}  {r['step_energy_j']:6.0f}  {r['step_elapsed_s']:6.0f}  {r['step_average_power_w']:10.3f}  {r['estimated_gflops_per_w']:.4f}")
by_id = {r["job_id"]:r for r in rows}
print("\n72x1 relative to 144x1:")
for field in ("gflops", "step_average_power_w", "estimated_gflops_per_w"):
    print(field, f"{100*(by_id['558659'][field]/by_id['558657'][field]-1):+.2f}%")
assert not any(r["benchmark_window_energy_verified"] for r in rows)


Job     GFLOP/s  Whole-job J  Step J  Step s  Avg step W  Estimated GFLOP/s/W
558650  134.942       273140  270439     257    1052.292  0.1282
558651  132.918       274050  269594     261    1032.927  0.1287
558653  134.293       269130  266029     256    1039.176  0.1292
558654  133.793       269963  266458     258    1032.783  0.1295
558656  134.894       271615  268882     256    1050.320  0.1284
558657  135.342       231331  228809     247     926.352  0.1461
558658  134.188       264250  261950     249    1052.008  0.1276
558659  131.806       216374  214062     255     839.459  0.1570
558660  133.809       267498  264883     258    1026.678  0.1303

72x1 relative to 144x1:
gflops -2.61%
step_average_power_w -9.38%
estimated_gflops_per_w +7.47%


## Results and takeaways

144×1: 135.342 GFLOP/s, 926.352 W step average, 0.1461 estimated GFLOP/s/W.
72×1: 131.806 GFLOP/s, 839.459 W step average, 0.1570 estimated GFLOP/s/W.
The 72×1 observation is 2.61% slower with a 7.47% higher efficiency estimate.
The 0.31% apparent speed improvement of 144×1 over the repeated 288×1 mean remains unconfirmed.

The exact-value table is used for this nine-run accounting reconciliation. Future long multi-node measurements must verify node coverage and >=1800 reported seconds. Slurm-step estimates remain the fallback; no efficiency objective is enabled.
